# Proyek Akhir Praktikum Big Data: TidarMart Data Platform
**Analitik Big Data End-to-End pada Resource Terbatas**

- **Nama:** Angger Naufal Abror
- **NPM:** 2505060020
- **VARIAN:** 0 (Pembatalan dan Logistik)

---

## Tahap 1 – Penyimpanan Terdistribusi dan Ingesti (HDFS)

### Tujuan
Memastikan seluruh dataset mentah (`transaksi`, `pelanggan`, `produk`, dan `stream_input`) sudah tersimpan di HDFS dengan struktur direktori yang terorganisir, serta memverifikasi ukuran, jumlah blok, dan faktor replikasi.

### Tuntunan Pengerjaan
- **1. Buat struktur direktori HDFS** menggunakan `!hdfs dfs -mkdir -p ...`.
- **2. Unggah seluruh data** dari `data_proyek/` ke HDFS menggunakan `!hdfs dfs -put -f ...`.
- **3. Verifikasi HDFS** menggunakan awalan `!` (misalnya `hdfs dfs -du -h` dan `hdfs fsck`) agar output bukti **Volume** tersimpan di notebook.

In [3]:
# 1. Buat struktur direktori di HDFS
!hdfs dfs -mkdir -p /tidarmart/raw/transaksi /tidarmart/raw/pelanggan /tidarmart/raw/produk /tidarmart/stream_input /tidarmart/curated /tidarmart/models

# 2. Unggah data ke HDFS 
!hdfs dfs -put -f data_proyek/transaksi/*.csv /tidarmart/raw/transaksi/
!hdfs dfs -put -f data_proyek/pelanggan.json /tidarmart/raw/pelanggan/
!hdfs dfs -put -f data_proyek/produk.json /tidarmart/raw/produk/
!hdfs dfs -put -f data_proyek/stream_input/*.csv /tidarmart/stream_input/

In [2]:
# Masih tahap 1

!hdfs dfs -ls -R /tidarmart

drwxr-xr-x   - angger supergroup          0 2026-11-26 14:41 /tidarmart/curated
drwxr-xr-x   - angger supergroup          0 2026-11-26 14:41 /tidarmart/models
drwxr-xr-x   - angger supergroup          0 2026-11-26 14:41 /tidarmart/raw
drwxr-xr-x   - angger supergroup          0 2026-11-26 14:41 /tidarmart/raw/pelanggan
-rw-r--r--   1 angger supergroup    6307375 2026-11-26 14:41 /tidarmart/raw/pelanggan/pelanggan.json
drwxr-xr-x   - angger supergroup          0 2026-11-26 14:41 /tidarmart/raw/produk
-rw-r--r--   1 angger supergroup     697939 2026-11-26 14:41 /tidarmart/raw/produk/produk.json
drwxr-xr-x   - angger supergroup          0 2026-11-26 14:41 /tidarmart/raw/transaksi
-rw-r--r--   1 angger supergroup   49075424 2026-11-26 14:41 /tidarmart/raw/transaksi/transaksi_part1.csv
-rw-r--r--   1 angger supergroup   49068616 2026-11-26 14:41 /tidarmart/raw/transaksi/transaksi_part2.csv
-rw-r--r--   1 angger supergroup   49070606 2026-11-26 14:41 /tidarmart/raw/transaksi/transaksi_part3.

In [5]:
print("List dan ukuran data folder stream_input:\n")
!hdfs dfs -du -h /tidarmart/stream_input
print("\nList dan ukuran data folder raw:\n")
!hdfs dfs -du -h /tidarmart/raw

List dan ukuran data folder stream_input:

479.7 K  479.7 K  /tidarmart/stream_input/stream_01.csv
479.6 K  479.6 K  /tidarmart/stream_input/stream_02.csv
479.9 K  479.9 K  /tidarmart/stream_input/stream_03.csv
478.7 K  478.7 K  /tidarmart/stream_input/stream_04.csv
479.1 K  479.1 K  /tidarmart/stream_input/stream_05.csv
479.1 K  479.1 K  /tidarmart/stream_input/stream_06.csv
479.4 K  479.4 K  /tidarmart/stream_input/stream_07.csv
479.5 K  479.5 K  /tidarmart/stream_input/stream_08.csv
479.3 K  479.3 K  /tidarmart/stream_input/stream_09.csv
479.3 K  479.3 K  /tidarmart/stream_input/stream_10.csv
479.5 K  479.5 K  /tidarmart/stream_input/stream_11.csv
479.1 K  479.1 K  /tidarmart/stream_input/stream_12.csv
479.3 K  479.3 K  /tidarmart/stream_input/stream_13.csv
479.5 K  479.5 K  /tidarmart/stream_input/stream_14.csv
479.0 K  479.0 K  /tidarmart/stream_input/stream_15.csv
479.0 K  479.0 K  /tidarmart/stream_input/stream_16.csv
479.6 K  479.6 K  /tidarmart/stream_input/stream_17.csv
479.7

###  Pertanyaan Konsep & Interpretasi Tahap 1
1. **Mengapa direktori `raw` (mentah) dan `curated` (bersih) dipisah?** *(Tulis penjelasanmu di sini)*
2. **Apa arti faktor replikasi = 1 pada VM pseudo-distributed dibanding klaster produksi nyata?** *(Tulis penjelasanmu di sini)*

1. **a. Untuk menjaga keaslian data:** Data di /tidarmart/raw adalah single source of truth dari semua format raw (mentah). Nah kalau  dicampur atau ditimpa, data asli bisa hilang atau rusak permanen.
**b. Memberikan Kemudahan re-run & audit:** Jika di tengah jalan aturan pembersihan (ETL) diubah atau ditemukan bug baru, kita bisa memproses ulang dari raw ke curated tanpa harus mengambil ulang data dari awal.
**c. Kesiapan pakai & efisiensi:** Data di curated sudah bersih, seragam, dan berformat Parquet sehingga proses analitik (Tahap 4) dan Machine Learning (Tahap 7) bisa langsung membacanya dengan cepat tanpa mengulang proses pembersihan.
2. Karena hanya ada 1 DataNode yang berjalan di dalam satu mesin VM, HDFS hanya menyimpan 1 salinan blok data (tidak ada duplikat di node lain). Artinya, tidak ada toleransi kesalahan (fault tolerance): jika disk VM rusak atau DataNode bermasalah, data bisa hilang. Konfigurasi ini hanya cocok untuk belajar/pengembangan pada resource terbatas.

## Tahap 2 – Profiling Kualitas Data (Veracity)

### Tujuan
Memuat data multi-format dari HDFS ke PySpark sesuai batasan memori, lalu mengidentifikasi seluruh masalah kualitas data (*Veracity*) dan variasi struktur (*Variety*).

### Tuntunan Pengerjaan
1. Buat `SparkSession` dengan konfigurasi minimum wajib (`local[2]`, `spark.driver.memory = 1536m`, `spark.sql.shuffle.partitions = 4`, `spark.sql.adaptive.enabled = true`).
2. Baca data dari `hdfs://localhost:9000/tidarmart/raw/...`. Gunakan `StructType` eksplisit untuk JSON (`pelanggan`, `produk`). Jika memakai `inferSchema` untuk CSV, catat *trade-off*-nya.
3. **Transaksi:** Hitung total baris, jumlah `order_id` unik, % nilai kosong per kolom, jumlah `harga_satuan` negatif, distribusi `qty` (kuantil untuk deteksi outlier), dan variasi penulisan `kota`.
4. **Pelanggan & Produk:** Tunjukkan persentase kelengkapan atribut produk per kategori dan jelaskan perlakuan field array `tags`.
5. **Aturan:** Dilarang `toPandas()` atau `collect()` > 10.000 baris.

In [1]:
# 1. Inisialisasi SparkSession dan pembacaan data dari HDFS
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, ArrayType
)

# Konfigurasi minimum wajib sesuai aturan proyek
spark = (SparkSession.builder
    .appName("TidarMart")
    .master("local[2]")
    .config("spark.driver.memory", "1536m")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate())

# Skema eksplisit JSON Pelanggan
schema_pelanggan = StructType([
    StructField("pelanggan_id", IntegerType(), True),
    StructField("nama", StringType(), True),
    StructField("kota", StringType(), True),
    StructField("tanggal_daftar", StringType(), True),
    StructField("membership", StringType(), True),
])

# Skema eksplisit JSON Produk
schema_produk = StructType([
    StructField("produk_id", IntegerType(), True),
    StructField("nama_produk", StringType(), True),
    StructField("kategori", StringType(), True),
    StructField("harga_dasar", IntegerType(), True),
    StructField("berat_kg", DoubleType(), True),
    StructField("garansi_bulan", IntegerType(), True),
    StructField("voltase", IntegerType(), True),
    StructField("ukuran", StringType(), True),
    StructField("bahan", StringType(), True),
    StructField("kadaluarsa_hari", IntegerType(), True),
    StructField("penerbit", StringType(), True),
    StructField("tags", ArrayType(StringType()), True),
])

# Baca ketiga sumber dari HDFS 
df_trx = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("hdfs://localhost:9000/tidarmart/raw/transaksi/"))

df_pelanggan = spark.read.schema(schema_pelanggan).json("hdfs://localhost:9000/tidarmart/raw/pelanggan/pelanggan.json")
df_produk = spark.read.schema(schema_produk).json("hdfs://localhost:9000/tidarmart/raw/produk/produk.json")

df_trx.printSchema()
df_trx.show(5, truncate=False)

26/11/26 22:26:04 WARN Utils: Your hostname, angger resolves to a loopback address: 127.0.1.1; using 192.168.1.14 instead (on interface wlp3s0)
26/11/26 22:26:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/26 22:26:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- pelanggan_id: integer (nullable = true)
 |-- produk_id: integer (nullable = true)
 |-- qty: double (nullable = true)
 |-- harga_satuan: double (nullable = true)
 |-- diskon_persen: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- berat_kg: double (nullable = true)
 |-- jarak_km: double (nullable = true)
 |-- lama_kirim_hari: integer (nullable = true)
 |-- status_pesanan: string (nullable = true)

+----------+-------------------+------------+---------+---+------------+-------------+-----------------+--------+--------+--------+---------------+--------------+
|order_id  |tanggal            |pelanggan_id|produk_id|qty|harga_satuan|diskon_persen|metode_pembayaran|kota    |berat_kg|jarak_km|lama_kirim_hari|status_pesanan|
+----------+-------------------+------------+---------+---+------------+-------------+-----------------+--------+--

In [2]:
# 2. Profiling data transaksi, pelanggan, dan produk
total_baris = df_trx.count()
unik_order = df_trx.select(F.countDistinct("order_id")).collect()[0][0]
duplikat = total_baris - unik_order
harga_negatif = df_trx.filter(F.col("harga_satuan") < 0).count()

print(f"Total baris       : {total_baris:,}")
print(f"Order ID unik     : {unik_order:,}")
print(f"Estimasi duplikat : {duplikat:,}")
print(f"Harga negatif     : {harga_negatif:,}")

# Presentase niali kosong
null_exprs = [
    F.round(F.count(F.when(F.col(c).isNull(), c)) / total_baris * 100, 3).alias(c)
    for c in df_trx.columns
]
print("\nPersentase nilai kosong (%) per kolom:")
df_trx.select(null_exprs).show(truncate=False)
# Kuantil qty untuk menemukan outlier
kuantil_qty = df_trx.approxQuantile("qty", [0.0, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999, 1.0], 0.001)
print("Kuantil qty [min, Q1, median, Q3, P95, P99, P99.9, max]:", kuantil_qty)
print("Jumlah qty > 10 (outlier):", df_trx.filter(F.col("qty") > 10).count())
# Variasi penulisan kota
print("\nVariasi penulisan kota:")
df_trx.groupBy("kota").count().orderBy("kota").show(50, truncate=False)
# 2. PROFILING PELANGGAN & PRODUK (VARIETY)
print(f"Total pelanggan: {df_pelanggan.count():,} | Total produk: {df_produk.count():,}")
atribut_opsional = ["garansi_bulan", "voltase", "ukuran", "bahan", "kadaluarsa_hari", "penerbit", "tags"]
expr_produk = [
    F.round(F.count(F.col(a)) / F.count("*") * 100, 1).alias(f"{a}_pct")
    for a in atribut_opsional
]
print("\nPersentase kelengkapan atribut produk per kategori (%):")
df_produk.groupBy("kategori").agg(*expr_produk).orderBy("kategori").show(truncate=False)
# Perlakuan field array tags
pct_tags = round(df_produk.filter(F.col("tags").isNotNull()).count() / df_produk.count() * 100, 2)
print(f"Produk yang memiliki tags: {pct_tags}%")
df_produk.select(F.explode("tags").alias("tag")).groupBy("tag").count().orderBy(F.desc("count")).show()

Total baris       : 2,010,000
Order ID unik     : 2,000,000
Estimasi duplikat : 10,000
Harga negatif     : 1,994

Persentase nilai kosong (%) per kolom:


+--------+-------+------------+---------+---+------------+-------------+-----------------+----+--------+--------+---------------+--------------+
|order_id|tanggal|pelanggan_id|produk_id|qty|harga_satuan|diskon_persen|metode_pembayaran|kota|berat_kg|jarak_km|lama_kirim_hari|status_pesanan|
+--------+-------+------------+---------+---+------------+-------------+-----------------+----+--------+--------+---------------+--------------+
|0.0     |0.0    |0.0         |0.0      |0.3|0.302       |0.0          |0.0              |0.0 |0.0     |0.0     |0.0            |0.0           |
+--------+-------+------------+---------+---+------------+-------------+-----------------+----+--------+--------+---------------+--------------+



Kuantil qty [min, Q1, median, Q3, P95, P99, P99.9, max]: [1.0, 2.0, 4.0, 6.0, 7.0, 7.0, 999.0, 999.0]


Jumlah qty > 10 (outlier): 1040

Variasi penulisan kota:


+----------+------+
|kota      |count |
+----------+------+
|KEBUMEN   |6466  |
|Kebumen   |239178|
|MAGELANG  |6294  |
|Magelang  |239018|
|PURWOREJO |6262  |
|Purworejo |238292|
|SEMARANG  |6311  |
|SOLO      |6228  |
|Semarang  |238900|
|Solo      |239228|
|TEMANGGUNG|6088  |
|Temanggung|238779|
|WONOSOBO  |6309  |
|Wonosobo  |237812|
|YOGYAKARTA|6224  |
|Yogyakarta|238177|
|kebumen   |6371  |
|magelang  |6361  |
|purworejo |6329  |
|semarang  |6247  |
|solo      |6303  |
|temanggung|6261  |
|wonosobo  |6231  |
|yogyakarta|6331  |
+----------+------+

Total pelanggan: 50,000 | Total produk: 5,000

Persentase kelengkapan atribut produk per kategori (%):
+------------+-----------------+-----------+----------+---------+-------------------+------------+--------+
|kategori    |garansi_bulan_pct|voltase_pct|ukuran_pct|bahan_pct|kadaluarsa_hari_pct|penerbit_pct|tags_pct|
+------------+-----------------+-----------+----------+---------+-------------------+------------+--------+
|Buku       

### Ringkasan Profil "Sebelum Pembersihan" (Tahap 2)
- **Total baris:** 2.010.000 (`order_id` unik: 2.000.000, jadi ada 10.000 data duplikat)
- **Nilai kosong:** `qty` 0,3% dan `harga_satuan` 0,302% (kolom lainnya 0%)
- **Harga negatif (`< 0`):** 1.994 baris
- **Outlier `qty`:** normalnya 1–7, tapi ada 1.040 baris yang nilainya `999`
- **Penulisan `kota`:** ada 24 variasi huruf besar/kecil untuk 8 kota
- **Variasi produk & `tags`:** tiap kategori punya atribut khusus masing-masing (misal Elektronik punya garansi & voltase, Fashion punya ukuran & bahan, Makanan punya kadaluarsa, Buku punya penerbit). Field `tags` hanya dimiliki 29,54% produk, dibaca sebagai array, lalu dipecah pakai `explode` untuk dihitung isinya.
- **Trade-off `inferSchema=True`:** lebih praktis karena tidak perlu tulis skema CSV manual, tapi Spark membaca file CSV 2 kali sehingga sedikit lebih lambat.

## Tahap 3 – ETL dan Penyimpanan Kolumnar (Parquet)

### Tujuan
Membersihkan data transaksi berdasarkan temuan Tahap 2, menambahkan kolom turunan, menyimpannya dalam format Parquet berpartisi di HDFS, serta membandingkan efisiensi CSV vs Parquet.

### Tuntunan Pengerjaan
1. Terapkan aturan pembersihan (deduplikasi, nilai kosong, harga negatif, outlier `qty`, normalisasi `kota`) beserta alasan masing-masing.
2. Tambahkan kolom turunan: `total_nilai`, `bulan`, `hari_dalam_minggu`, `flag_batal`.
3. Simpan ke `hdfs://localhost:9000/tidarmart/curated/...` dalam format Parquet dengan `partitionBy` pilihanmu.
4. Bandingkan ukuran HDFS dan waktu eksekusi 1 query agregasi yang sama antara CSV vs Parquet.
5. Tampilkan laporan profil "Sesudah Pembersihan" berdampingan dengan "Sebelum Pembersihan".

In [3]:
# 1. Pembersihan data, penambahan kolom turunan, dan simpan ke Parquet di HDFS
from pyspark.sql import functions as F

df_clean = (df_trx
    .dropDuplicates(["order_id"])
    .filter(F.col("qty").isNotNull() & F.col("harga_satuan").isNotNull())
    .filter(F.col("harga_satuan") > 0)
    .filter((F.col("qty") >= 1) & (F.col("qty") <= 10))
    .withColumn("kota", F.initcap(F.trim(F.col("kota"))))
)

df_curated = (df_clean
    .withColumn("total_nilai", F.round(F.col("qty") * F.col("harga_satuan") * (1 - F.col("diskon_persen") / 100.0), 2))
    .withColumn("bulan", F.month(F.col("tanggal")))
    .withColumn("hari_dalam_minggu", F.dayofweek(F.col("tanggal")))
    .withColumn("flag_batal", F.when(F.col("status_pesanan") == "Dibatalkan", 1).otherwise(0))
)
df_curated.select("order_id", "kota", "total_nilai", "bulan", "hari_dalam_minggu", "flag_batal").show(5)

# Simpan kedalam parquet
output_parquet = "hdfs://localhost:9000/tidarmart/curated/transaksi_parquet"
df_curated.write.mode("overwrite").partitionBy("bulan").parquet(output_parquet)
print("Berhasil simpan Parquet ke:", output_parquet)

+----------+----------+-----------+-----+-----------------+----------+
|  order_id|      kota|total_nilai|bulan|hari_dalam_minggu|flag_batal|
+----------+----------+-----------+-----+-----------------+----------+
|ORD1000000|  Magelang|   384000.0|    3|                2|         0|
|ORD1000009|   Kebumen|    50000.0|    3|                7|         0|
|ORD1000010|      Solo|   760000.0|    3|                2|         0|
|ORD1000013|  Semarang|  2137500.0|    4|                7|         0|
|ORD1000016|Temanggung|  3000000.0|    5|                5|         0|
+----------+----------+-----------+-----+-----------------+----------+
only showing top 5 rows



[Stage 48:===========================================>              (3 + 1) / 4]

Berhasil simpan Parquet ke: hdfs://localhost:9000/tidarmart/curated/transaksi_parquet


In [4]:
# 2. Perbandingan CSV vs Parquet & profil sesudah pembersihan
import time

df_parquet = spark.read.parquet("hdfs://localhost:9000/tidarmart/curated/transaksi_parquet")

# Uji waktu query agregasi yang sama: CSV vs Parquet
t0 = time.time()
df_trx.filter(F.col("status_pesanan") == "Selesai").groupBy("metode_pembayaran").agg(F.sum(F.col("qty") * F.col("harga_satuan"))).collect()
waktu_csv = time.time() - t0

t0 = time.time()
df_parquet.filter(F.col("status_pesanan") == "Selesai").groupBy("metode_pembayaran").agg(F.sum("total_nilai")).collect()
waktu_parquet = time.time() - t0

print(f"Waktu CSV     : {waktu_csv:.2f} detik")
print(f"Waktu Parquet : {waktu_parquet:.2f} detik")

# Bandingkan ukuran di HDFS
!hdfs dfs -du -h -s /tidarmart/raw/transaksi
!hdfs dfs -du -h -s /tidarmart/curated/transaksi_parquet

# Cek ulang profil sesudah pembersihan
total_bersih = df_parquet.count()
unik_bersih = df_parquet.select(F.countDistinct("order_id")).collect()[0][0]
print(f"Total bersih: {total_bersih:,} | Unik: {unik_bersih:,} | Duplikat: {total_bersih - unik_bersih}")
print("Null qty/harga:", df_parquet.filter(F.col("qty").isNull() | F.col("harga_satuan").isNull()).count())
print("Harga <= 0    :", df_parquet.filter(F.col("harga_satuan") <= 0).count())
print("Outlier qty>10:", df_parquet.filter(F.col("qty") > 10).count())
print("Kota unik     :", df_parquet.select("kota").distinct().count())

Waktu CSV     : 1.45 detik
Waktu Parquet : 0.68 detik
187.2 M  187.2 M  /tidarmart/raw/transaksi
54.5 M  163.6 M  /tidarmart/curated/transaksi_parquet


Total bersih: 1,984,973 | Unik: 1,984,973 | Duplikat: 0
Null qty/harga: 0
Harga <= 0    : 0
Outlier qty>10: 0
Kota unik     : 8


### Ringkasan & Perbandingan Tahap 3

**1. Alasan Pembersihan & Pemilihan Partisi:**
- **Duplikat:** Dibuang pakai `dropDuplicates("order_id")` supaya satu pesanan tidak terhitung dua kali.
- **Nilai kosong & harga negatif:** Dibuang karena jumlahnya sedikit (<1%) dan tanpa `qty`/harga yang valid, `total_nilai` tidak bisa dihitung.
- **Outlier `qty` (`999`):** Dibuang (`qty <= 10`) karena dari pengecekan kuantil, jumlah pembelian normal hanya 1–7 barang.
- **Nama `kota`:** Dirapikan pakai `initcap` dan `trim` supaya huruf besar/kecilnya seragam menjadi 8 kota.
- **Pilih `partitionBy("bulan")`:** Karena datanya 6 bulan (Jan–Jun), jadi terbagi rapi menjadi 6 folder dan lebih cepat saat query per bulan.

**2. Perbandingan CSV vs Parquet:**
- **CSV:** ukuran 187,2 MB, waktu query agregasi **1,45 detik**.
- **Parquet:** ukuran **54,5 MB**, waktu query agregasi **0,68 detik** (ukuran jauh lebih kecil dan eksekusi lebih dari 2x lebih cepat).

**3. Kondisi Sebelum vs Sesudah Pembersihan:**
- **Total baris:** dari 2.010.000 menjadi **1.984.973** (`order_id` unik: 1.984.973)
- **Duplikat:** dari 10.000 menjadi **0**
- **Nilai kosong (`qty` / `harga_satuan`):** dari 0,3% / 0,302% menjadi **0**
- **Harga negatif:** dari 1.994 menjadi **0**
- **Outlier `qty > 10`:** dari 1.040 menjadi **0**
- **Variasi `kota`:** dari 24 variasi menjadi **8 kota unik**

*Kesimpulan: Data sudah bersih dari duplikat/nilai aneh, ukuran file di HDFS turun drastis dari 187,2 MB jadi 54,5 MB, dan query berjalan lebih dari 2x lebih cepat.*

## Tahap 4 – Analitik Batch (DataFrame, Join, Window, Spark SQL)

### Tujuan
Menggali wawasan bisnis (*Value*) dari data Parquet bersih melalui 5 analisis inti (A–E) dan 3 Pertanyaan Bisnis Khusus **VARIAN 0** (F–H).

### Tuntunan Pengerjaan (contekan)
- **A:** Join transaksi, produk, dan pelanggan $\rightarrow$ omzet per kategori per kota (jelaskan strategi join/broadcast).
- **B:** Tren omzet bulanan & pertumbuhan dibanding bulan sebelumnya (`lag`).
- **C:** Top 5 produk terlaris di setiap kota berdasarkan omzet (`rank` / `dense_rank`).
- **D:** Omzet kumulatif (*running total*) per kategori & rata-rata bergerak 7 hari.
- **E:** Tulis ulang salah satu analisis A–D memakai Spark SQL (`createOrReplaceTempView`), buktikan hasil identik, dan bandingkan `explain()`.
- **F (Varian 0):** Tingkat pembatalan menurut `lama_kirim_hari` & `metode_pembayaran`; temukan kombinasi `(kota, metode)` dengan pembatalan tertinggi.
- **G (Varian 0):** Kelompokkan `jarak_km` (bucket) untuk melihat di jarak berapa pembatalan mulai melonjak + sajikan grafik.
- **H (Varian 0):** Estimasi omzet hilang akibat pembatalan per kota + 3 kota paling merugi.
- **Wajib:** Tutup setiap analisis dengan **1 kalimat interpretasi bisnis**.

In [ ]:
# Analisis Inti A, B, C, D, dan E (Spark SQL + explain)

In [ ]:
# Pertanyaan Bisnis Khusus VARIAN 0: F, G (dengan grafik), dan H

### Interpretasi Bisnis Tahap 4 (A–H)
- **Strategi Join (A) & Perbandingan `explain()` (E):** *(Tulis di sini)*
- **Ringkasan 1 Kalimat Interpretasi Bisnis per Analisis (A–H):** *(Tulis di sini)*

## Tahap 5 – Penyimpanan NoSQL (MongoDB)

### Tujuan
Memodelkan data semi-terstruktur dan ringkasan profil pelanggan ke MongoDB (`tidarmart`), serta menguji optimasi indeks dan operasi query/aggregasi.

### Tuntunan Pengerjaan
1. Muat `pelanggan.json` & `produk.json` ke MongoDB (pastikan tipe data Python native, bukan NumPy).
2. Buat ringkasan "profil pelanggan 360" dari Spark dan simpan ke koleksi `profil_pelanggan` via `insert_many` per batch (misal 5.000 dokumen).
3. Buat indeks pada `pelanggan_id` dan bandingkan `explain()` dengan vs tanpa indeks.
4. Tunjukkan minimal 5 operasi: `$gt`/`$in`, query array `tags`, `$exists`, `update_one`, dan aggregation pipeline (`$group`/`$match`/`$sort`).

In [ ]:
# Ingesti ke MongoDB, profil_pelanggan (batch insert_many), indexing + explain, dan 5 operasi

### Pertanyaan Konsep & Interpretasi Tahap 5
1. **Mengapa produk dengan atribut bervariasi cocok disimpan sebagai dokumen?** *(Tulis di sini)*
2. **Kapan data sebaiknya tetap di Parquet/HDFS dan kapan di MongoDB? (Minimal 2 skenario)** *(Tulis di sini)*
3. **Interpretasi:** *(Ringkas dampak indeks terhadap waktu/rencana eksekusi)*

## Tahap 6 – Pemrosesan Real-Time (Structured Streaming)

### Tujuan
Membangun pipeline streaming untuk menghitung agregasi berbasis window waktu dan mendeteksi anomali transaksi secara real-time (*Velocity*).

### Tuntunan Pengerjaan
1. Baca `stream_input/` menggunakan `readStream` dengan skema eksplisit (`StructType`) dan `maxFilesPerTrigger = 1` (atau `2`).
2. Hitung agregasi window waktu (misal 1 jam per kota) ke memory sink / file sink.
3. Terapkan deteksi anomali (`total_nilai` di atas ambang statistik batch, misal persentil 99,9) ke sink terpisah.
4. Buktikan hasil bertambah saat berkas stream baru masuk (tangkap sebelum & sesudah), lalu hentikan seluruh query dengan `query.stop()`.

In [ ]:
# Structured Streaming: readStream, window aggregation, deteksi anomali, verifikasi sebelum/sesudah, dan query.stop()

### Penjelasan Desain & Interpretasi Tahap 6
- **Alasan pilihan sumber stream (HDFS vs lokal) & Output Mode (`append`/`update`/`complete`):** *(Tulis di sini)*
- **Interpretasi:** *(Ringkas bukti Velocity dari hasil sebelum & sesudah berkas baru masuk)*

## Tahap 7 – Machine Learning dengan MLlib

### Tujuan
Membangun, mengevaluasi, meningkatkan, dan menyimpan 2 model MLlib menggunakan `Pipeline` pada data uji yang bersih.

### Tuntunan Pengerjaan
1. Kerjakan **2 dari 3 model** (disarankan **Model 1: Klasifikasi Pembatalan** karena dipakai di Tahap 8, plus **Model 2: Regresi** atau **Model 3: K-Means**).
2. Batas Random Forest: maks `numTrees = 50`, `maxDepth = 8`. Tangani *class imbalance* (Model 1) / cegah *data leakage* (Model 2).
3. Lakukan 1 eksperimen peningkatan dan sajikan perbandingan metriknya dalam tabel.
4. Simpan setiap `PipelineModel` ke HDFS (`/tidarmart/models/...`) dan buktikan `PipelineModel.load` menghasilkan prediksi identik.

In [ ]:
# Model 1 (Klasifikasi Pembatalan) + penanganan imbalance + simpan & load dari HDFS

In [ ]:
# Model kedua pilihanmu (Regresi atau K-Means) + eksperimen peningkatan

### Evaluasi Model, Eksperimen & Interpretasi Tahap 7
- **Mengapa Accuracy saja menyesatkan pada kelas tidak seimbang (Model 1):** *(Tulis di sini)*
- **Tabel Perbandingan Metrik & Hasil Eksperimen Peningkatan:** *(Tulis di sini)*
- **Interpretasi:** *(Ringkas performa model dan kesiapan dipakai ulang)*

## Tahap 8 – Integrasi, Optimasi Sumber Daya, dan Penutup

### Tujuan
Menyatukan pipeline batch + ML + streaming, menguji optimasi resource Spark, serta merangkum temuan teknis dan rekomendasi bisnis.

### Tuntunan Pengerjaan
1. **Integrasi:** Muat model klasifikasi pembatalan dari HDFS, terapkan pada data stream, dan tulis pesanan berisiko tinggi ke sink terpisah.
2. **Studi Sumber Daya:** Jalankan 1 pekerjaan berat dengan 2 konfigurasi berbeda (misal `shuffle.partitions = 4 vs 200` atau `cache vs tanpa cache`), bandingkan waktu eksekusi, dan amati **Spark UI (port 4040)**.

In [ ]:
# 1. Integrasi Model HDFS ke Structured Streaming
# 2. Studi Sumber Daya (perbandingan 2 konfigurasi)

### Penutup: Spark UI, Troubleshooting Log, Rekomendasi & Peningkatan Skala
1. **Temuan Spark UI (port 4040) & Tabel Studi Sumber Daya:** *(Tulis di sini)*
2. **Troubleshooting Log (minimal 3 masalah nyata + solusi):** *(Tulis di sini)*
3. **Tiga Rekomendasi Bisnis Berbasis Bukti:** *(Tulis di sini)*
4. **Rencana Peningkatan Skala (1.000× lebih besar di klaster sungguhan):** *(Tulis di sini)*